# LLM Deployment Landscape: Selection Matrix & Cost-Benefit Analysis

**My Objective:** I am evaluating and routing Large Language Models (LLMs) based on hard business constraints (data privacy, latency limits), operational costs, and inference speed. I built this analysis to determine the optimal model deployment across three distinct business scenarios: Public FAQ, PII Document Analysis, and Live Agent Assist.

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client

ollama = ollama_client()
try:
    groq = chat_client()                      # Groq API
except Exception as e:                        
    groq = None
    print("no hosted client:", str(e)[:80])
    
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %


## 1. Model Catalog & Baseline Economics
I loaded an indicative catalog of available models to establish a baseline for context windows, pricing (USD per million tokens), and operational modes (hosted API, local, or self-hosted GPU).

In [2]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

OVERRIDES = {
    "groq-llama31-8b": {"price_out": 0.08},      
}

model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

**My Observation — Self-Hosting Economics:** When I run open-weight models locally or on-premise, token generation is technically free. However, I noted that the operational baseline is not zero; our ongoing expenses simply shift entirely to fixed monthly costs for physical hardware or leased GPU cloud infrastructure.

## 2. Workload Scenarios & Cost Projections
I modeled the monthly financial impact of running these models against three distinct workloads with varying volumes:
* **FAQ:** Cheap, high-volume (20,000 calls/day), no sensitive data.
* **Documents:** Moderate volume (2,000 calls/day), high token count, contains Customer PII (Data must remain in-house).
* **Agent Assist:** High volume (30,000 calls/day), strict sub-3-second latency limit.

In [4]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))


faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        8,460/month
   gpt-oss-120b on Groq                           Rs       16,920/month
   Qwen3 32B on Groq                              Rs       26,282/month
   Llama 4 Scout 17B-16E on Groq                  Rs       11,280/month
   Llama 3.1 8B Instant on Groq                   Rs        4,286/month
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       65,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      188,000/month
   Frontier flagship (closed, vendor API)         Rs      620,400/month
   Frontier mid-tier (closed, vendor API)         Rs      248,160/month
   Frontier economy tier (closed, vendor API)     Rs       62,040/month

documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq      

**My Observation — Premium Tier ROI:** I calculated that for basic FAQ workloads, the frontier flagship model costs 281 times more than `gpt-oss-20b`. This massive premium is financially unjustifiable for low-complexity tasks. I recommend we only utilize flagship models for high-risk workloads where an AI error would result in significant financial loss.

## 3. Live Latency Benchmarking (TTFT vs. TPS)
Catalog numbers are theoretical. To measure real-world performance, I sent the same prompt to Groq and Ollama, tracking Time to First Token (TTFT) and Tokens Per Second (TPS) directly.

In [5]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

MEASURED = {
    "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)

Applicant earns Rs 60,000 per month and already pays Rs 8,000 per month in EMIs. She asks for a Rs 8,00,000 personal loan at 11.5% per annum for 60 months. Compute the monthly EMI, then say whether (EMI + existing EMIs) is under 50% of her income. Answer in three short lines. 

--- groq: openai/gpt-oss-20b
first token 1.54 s | total 1.64 s | 1004 tokens | 613.4 tokens/s
arithmetic check: {'emi_ok': True, 'ratio_ok': True, 'score_out_of_2': 2}
EMI ≈ Rs 17,600 per month  
Total debt payments (new + existing) ≈ Rs 25,600 per month  
This is about 42.7 % of her Rs 60,000 income – below the 50 % cut‑off. 

--- ollama: llama3.2:3b
first token 2.89 s | total 49.28 s | 123 tokens | 2.5 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
To calculate the monthly EMI, we use the formula: EMI = (P * R * T) / (1 - (1 + R)^(-T))
EMI = (800000 * 11.5/100 * 60/12) = Rs 51,150

The total EMIs will be Rs 8,000 (existing) + Rs 51,150 (personal loan) = Rs 59,150
This exce

**My Observation — Defining "Fast":** I observed that `gpt-oss-20b` on Groq achieved a faster Time to First Token (TTFT), while the frontier economy tier generated more tokens per second overall. For live customer-facing applications, I concluded that "fast" must be defined by TTFT, as a rapid initial response is critical to retaining customer attention.

## 4. Selection Matrix & Weighted Scoring
I aggressively eliminated models if they failed hard constraints (e.g., PII leakage, latency limits). I then ranked the survivors using a weighted score of cost, speed, and quality.

In [ ]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")


=== faq: Public product FAQ chatbot   weights {'cost': 0.6, 'latency': 0.2, 'quality': 0.2}
  1. gpt-oss-20b on Groq                            score  90.0   Rs      8,460/month   1.5 s   Q3
  2. Llama 4 Scout 17B-16E on Groq                  score  86.0   Rs     11,280/month   1.5 s   Q3
  3. gpt-oss-120b on Groq                           score  83.1   Rs     16,920/month   2.0 s   Q4
  4. Qwen3 32B on Groq                              score  72.0   Rs     26,282/month   2.0 s   Q3
  5. Frontier economy tier (closed, vendor API)     score  62.2   Rs     62,040/month   1.5 s   Q3
  6. gpt-oss-20b self-hosted on a smaller GPU serve score  57.0   Rs     65,800/month   2.5 s   Q3
  7. gpt-oss-120b self-hosted on your own GPU serve score  45.0   Rs    188,000/month   3.0 s   Q4
  8. Frontier mid-tier (closed, vendor API)         score  38.9   Rs    248,160/month   3.5 s   Q4
  9. Frontier flagship (closed, vendor API)         score  20.0   Rs    620,400/month   6.0 s   Q5
  x  Llama 3.1 8

**My Observation — Security vs. Speed Trade-off:** Because the 'documents' workload contains PII, I recognized we cannot rely on third-party flagship APIs. To prevent data leakage, I was forced to deploy an open-weight model on-premises. Guaranteeing data sovereignty means we must deliberately sacrifice the faster inference speeds and advanced reasoning capabilities offered by the cloud tier.

## 5. Final Deployment Decisions
Based on my hard constraints and matrix scoring, I have designated the routing architecture as follows:

* **Public FAQ Chatbot:** `gpt-oss-20b` on Groq. Because FAQ tasks have low complexity, I found the open-weight model provides sufficient quality. I avoided the flagship model entirely to eliminate unnecessary costs.
* **Document Analysis (PII):** `gpt-oss-20b` self-hosted on a smaller GPU server. I deployed this strictly on local infrastructure to ensure data privacy and isolated inference.
* **Live Agent Assist:** `gpt-oss-20b` on Groq. I chose this specifically because it is optimized for low TTFT and cost-efficient inference, fitting safely within our strict latency budgets.

In [6]:
# Output matrix data to markdown file for CI/CD tracking
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   
lines = ["# Model Selection Matrix Output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
OUT.write_text("\n".join(lines), encoding="utf-8")
print("Matrix analysis saved to", OUT)

Matrix analysis saved to c:\Users\HP\Desktop\FSDS\genaicourse\genai-agents-course\module01\s10_matrix_output.md
